# 09. SQL analytics on the scored portfolio

**Objective.** Answer portfolio questions with SQL on the model's scored applicants, the way an analytics team would.

**Business question.** How is risk and expected loss distributed across the portfolio, and what does each approval cutoff do to volume and value?

**Why this analysis is needed.** Analysts and underwriting managers query databases, not notebooks. The scored validation and test applicants are loaded into SQLite (standard library, portable SQL: CTEs, window functions, `NTILE`, `RANK`), and every query lives in `sql/` as a separate file. All queries below use the validation split.

*Educational project. Not a lending policy.*

In [1]:
import joblib
from src import config, sql_runner
from src.data import load_modelling_table

table = load_modelling_table()
bundle = joblib.load(config.MODELS_DIR / "pd_model.joblib")
frame = sql_runner.scored_applicants(table, bundle)
path = sql_runner.build_database(frame)
print(path.name, len(frame), "rows;", frame["split"].value_counts().to_dict())

portfolio.db 123005 rows; {'test': 61503, 'valid': 61502}


## Portfolio summary by contract type

**Question.** How much exposure and expected loss does each product carry?

```sql
SELECT
    contract_type,
    COUNT(*)                                        AS applicants,
    ROUND(AVG(target), 4)                           AS default_rate,
    ROUND(AVG(pd), 4)                               AS mean_predicted_pd,
    ROUND(SUM(credit) / 1e6, 1)                     AS exposure_millions,
    ROUND(SUM(expected_loss) / 1e6, 1)              AS expected_loss_millions,
    ROUND(SUM(expected_loss) / SUM(credit), 4)      AS expected_loss_rate
FROM applicants
WHERE split = 'valid'
GROUP BY contract_type
ORDER BY exposure_millions DESC;
```

In [2]:
print(sql_runner.run_query('01_portfolio_summary.sql').to_string(index=False))

  contract_type  applicants  default_rate  mean_predicted_pd  exposure_millions  expected_loss_millions  expected_loss_rate
     Cash loans       55650        0.0834             0.0836            34909.0                  1200.8              0.0344
Revolving loans        5852        0.0555             0.0571             1911.6                    41.5              0.0217


**Result.** Cash loans carry 34,909 million of the 36,820 million exposure (95%) and a 3.44% expected-loss rate; revolving loans are 5% of exposure at 2.17%. **Interpretation.** The portfolio is almost entirely cash loans, so portfolio results are cash-loan results. Predicted PD matches the actual rate in both products (8.36% vs 8.34%; 5.71% vs 5.55%).

## Default rate by risk band

**Question.** Do bands separate risk, and how much of the defaulting sits in each?

```sql
SELECT
    band,
    COUNT(*)                                                   AS applicants,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1)         AS pct_of_applicants,
    ROUND(AVG(target), 4)                                      AS default_rate,
    ROUND(100.0 * SUM(target) / SUM(SUM(target)) OVER (), 1)   AS pct_of_defaulters
FROM applicants
WHERE split = 'valid'
GROUP BY band
ORDER BY band_order;
```

In [3]:
print(sql_runner.run_query('02_default_by_band.sql').to_string(index=False))

     band  applicants  pct_of_applicants  default_rate  pct_of_defaulters
      Low       18121               29.5        0.0172                6.3
 Moderate       16841               27.4        0.0431               14.6
 Elevated       10933               17.8        0.0798               17.6
     High        7500               12.2        0.1323               20.0
Very high        8107               13.2        0.2545               41.6


**Result.** The Very high band is 13.2% of applicants but 41.6% of defaulters; Low is 29.5% of applicants and 6.3% of defaulters. **Interpretation.** Bands are ordered and concentrate the defaulters; this reproduces the pandas table from Notebook 08 exactly.

## Riskiest income-type and age segments

**Question.** Which segments (at least 300 applicants) default most?

```sql
WITH segments AS (
    SELECT
        income_type,
        age_band,
        COUNT(*)            AS applicants,
        AVG(target)         AS default_rate,
        AVG(pd)             AS mean_pd
    FROM applicants
    WHERE split = 'valid'
    GROUP BY income_type, age_band
    HAVING COUNT(*) >= 300
)
SELECT
    income_type,
    age_band,
    applicants,
    ROUND(default_rate, 4)                    AS default_rate,
    ROUND(mean_pd, 4)                         AS mean_pd,
    RANK() OVER (ORDER BY default_rate DESC)  AS risk_rank
FROM segments
ORDER BY risk_rank
LIMIT 10;
```

In [4]:
print(sql_runner.run_query('03_segment_ranking.sql').to_string(index=False))

         income_type age_band  applicants  default_rate  mean_pd  risk_rank
             Working [20, 30)        5636        0.1253   0.1284          1
             Working [30, 40)       10364        0.1071   0.1022          2
Commercial associate [20, 30)        2668        0.0926   0.1048          3
Commercial associate [30, 40)        4627        0.0843   0.0825          4
       State servant [20, 30)         630        0.0841   0.0942          5
             Working [40, 50)        9523        0.0831   0.0838          6
           Pensioner [40, 50)         353        0.0765   0.0679          7
       State servant [30, 40)        1394        0.0710   0.0634          8
             Working [50, 60)        5583        0.0693   0.0714          9
Commercial associate [50, 60)        2410        0.0668   0.0568         10


**Result.** The riskiest large segments are working applicants aged 20-29 (12.5% default, 5,636 applicants) and 30-39 (10.7%). Predicted PD tracks the actual rate in each. **Interpretation.** Risk is concentrated among younger working-income applicants, which is consistent with the age pattern in Notebook 02. These segments are described for analytics; using age in a decision is the question raised in D31.

## Expected-loss concentration by risk decile

**Question.** How concentrated is expected loss in the riskiest applicants?

```sql
WITH ranked AS (
    SELECT
        expected_loss,
        target,
        NTILE(10) OVER (ORDER BY pd DESC) AS risk_decile
    FROM applicants
    WHERE split = 'valid'
),
by_decile AS (
    SELECT
        risk_decile,
        COUNT(*)            AS applicants,
        SUM(expected_loss)  AS expected_loss,
        AVG(target)         AS default_rate
    FROM ranked
    GROUP BY risk_decile
)
SELECT
    risk_decile,
    applicants,
    ROUND(default_rate, 4)                                                           AS default_rate,
    ROUND(100.0 * expected_loss / SUM(expected_loss) OVER (), 1)                     AS pct_of_expected_loss,
    ROUND(100.0 * SUM(expected_loss) OVER (ORDER BY risk_decile) / SUM(expected_loss) OVER (), 1) AS cumulative_pct
FROM by_decile
ORDER BY risk_decile;
```

In [5]:
print(sql_runner.run_query('04_loss_concentration.sql').to_string(index=False))

 risk_decile  applicants  default_rate  pct_of_expected_loss  cumulative_pct
           1        6151        0.2806                  32.5            32.5
           2        6151        0.1504                  18.0            50.5
           3        6150        0.1117                  12.7            63.2
           4        6150        0.0751                   9.8            73.1
           5        6150        0.0593                   7.7            80.8
           6        6150        0.0454                   6.1            86.9
           7        6150        0.0324                   4.9            91.8
           8        6150        0.0239                   3.9            95.7
           9        6150        0.0176                   2.7            98.4
          10        6150        0.0109                   1.6           100.0


**Result.** The riskiest 10% of applicants hold 32.5% of expected loss, the riskiest 20% hold 50.5%, and the riskiest 40% hold 73.1%. **Interpretation.** Loss is strongly concentrated, so decisions about the top deciles matter far more than about the rest.

## Approval cutoffs

**Question.** What does each cutoff do to approval, default rate and net value (LGD 45%, margin 8%)?

```sql
WITH cutoffs(cutoff) AS (
    VALUES (0.06), (0.10), (0.16), (0.20), (1.00)
)
SELECT
    c.cutoff,
    ROUND(AVG(CASE WHEN a.pd <= c.cutoff THEN 1.0 ELSE 0 END), 3)                          AS approval_rate,
    ROUND(1.0 * SUM(CASE WHEN a.pd <= c.cutoff THEN a.target END)
          / SUM(CASE WHEN a.pd <= c.cutoff THEN 1 END), 4)                                 AS default_rate_approved,
    ROUND(SUM(CASE WHEN a.pd <= c.cutoff AND a.target = 0 THEN a.credit * 0.08
                   WHEN a.pd <= c.cutoff AND a.target = 1 THEN -a.credit * 0.45
                   ELSE 0 END) / 1e6, 1)                                                   AS net_value_millions
FROM applicants a
CROSS JOIN cutoffs c
WHERE a.split = 'valid'
GROUP BY c.cutoff
ORDER BY c.cutoff;
```

In [6]:
print(sql_runner.run_query('05_cutoff_policy.sql').to_string(index=False))

 cutoff  approval_rate  default_rate_approved  net_value_millions
   0.06          0.568                 0.0297              1411.3
   0.10          0.746                 0.0416              1643.8
   0.16          0.868                 0.0543              1691.8
   0.20          0.911                 0.0603              1669.7
   1.00          1.000                 0.0807              1463.8


**Result.** Net value is 1,691.8 million at cutoff 0.16 and 1,463.8 million when approving everyone, identical to the pandas figures in Notebook 07. **Interpretation.** Two independent implementations (pandas and SQL) agree, which is the reason to run this query at all: it checks the policy logic. Cutoffs 0.10 and 0.20 are within 3% of the best value.

## Credit-history groups

**Question.** Does past behaviour separate risk in plain group-by terms?

```sql
SELECT
    CASE
        WHEN bureau_credit_count = 0 THEN 'no bureau record'
        WHEN inst_late_share IS NULL THEN 'no instalment history'
        WHEN inst_late_share = 0 THEN 'never late'
        WHEN inst_late_share < 0.2 THEN 'late on under 20%'
        ELSE 'late on 20% or more'
    END                              AS history_group,
    COUNT(*)                         AS applicants,
    ROUND(AVG(target), 4)            AS default_rate,
    ROUND(AVG(pd), 4)                AS mean_pd
FROM applicants
WHERE split = 'valid'
GROUP BY history_group
ORDER BY default_rate;
```

In [7]:
print(sql_runner.run_query('06_history_signal.sql').to_string(index=False))

        history_group  applicants  default_rate  mean_pd
no instalment history        2706        0.0480   0.0631
           never late       23004        0.0643   0.0647
    late on under 20%       19986        0.0836   0.0812
     no bureau record        8903        0.1011   0.1011
  late on 20% or more        6903        0.1136   0.1166


**Result.** Default rate rises from 6.4% for applicants never late, to 8.4% for those late on under 20% of instalments, to 11.4% for those late on 20% or more. Applicants with no bureau record default at 10.1%. Applicants with no earlier instalments default at 4.8%. **Interpretation.** Past lateness is a plain, explainable signal, and the model's mean predictions follow it. The groups are checked in this order (no bureau record first), so the no-instalment-history group only contains applicants who have a bureau record; that is a modelling detail to keep in mind when reading the group sizes.

## Next step

The Streamlit app reads the aggregates exported from these analyses; the AI Risk Analyst sits on top of a single applicant's model output.